In [ ]:
# ============================================================
# Cell 1
# ============================================================
import torch

if torch.cuda.is_available():
    gpu  = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu}")
    print(f"   VRAM: {vram:.1f} GB")
    if vram < 14:
        print("VRAM < 14GB")
else:
    raise RuntimeError("❌ Không có GPU!")

torch.cuda.empty_cache()
print(f"PyTorch: {torch.__version__} | CUDA: {torch.version.cuda}")

# Install packages
!pip install -q transformers==4.38.0 underthesea py_vncorenlp tabulate tqdm scikit-learn sentencepiece
print("Dependencies installed")

GPU: Tesla T4
   VRAM: 15.6 GB
PyTorch: 2.10.0+cu128 | CUDA: 12.8
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.1/131.1 kB 3.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.5/8.5 MB 75.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.3/7.3 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 35.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 108.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 70.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 30.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.2.3 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.38.0 which is incompatible.
Dependencies installed


In [2]:
# ============================================================
# Cell 2
# ============================================================
import os, sys

REPO_URL    = "https://github.com/vudinhminh08/NLP-project-master-study.git"
REPO_BRANCH = "master"
PROJECT_DIR = "/kaggle/working/absa-project"

# Clone (hoặc pull nếu đã có)
if not os.path.exists(PROJECT_DIR):
    print(f"Cloning {REPO_URL} (branch: {REPO_BRANCH})...")
    !git clone --branch {REPO_BRANCH} --depth=1 {REPO_URL} {PROJECT_DIR}
    print("Clone completed")
else:
    print(f"Repo đã tồn tại tại {PROJECT_DIR} — pulling latest...")
    !cd {PROJECT_DIR} && git pull origin {REPO_BRANCH}

# Chuyển vào thư mục project
os.chdir(PROJECT_DIR)
print(f"Working dir: {os.getcwd()}")

# Tạo thư mục cần thiết
for d in ["data", "outputs/models", "outputs/results", "outputs/eda"]:
    os.makedirs(d, exist_ok=True)

# Thêm Python paths
sys.path.insert(0, "code/data_processing")
sys.path.insert(0, "code/phobert")

# Kiểm tra cấu trúc repo
print("\nCấu trúc repo:")
!ls -la
!ls code/data_processing/ code/phobert/

Cloning https://github.com/vudinhminh08/NLP-project-master-study.git (branch: master)...
Cloning into '/kaggle/working/absa-project'...
remote: Enumerating objects: 118, done.
remote: Counting objects: 100% (118/118), done.
remote: Compressing objects: 100% (109/109), done.
remote: Total 118 (delta 11), reused 63 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (118/118), 4.71 MiB | 16.57 MiB/s, done.
Resolving deltas: 100% (11/11), done.
Clone completed
Working dir: /kaggle/working/absa-project

Cấu trúc repo:
total 48
drwxr-xr-x 9 root root 4096 Apr 16 18:56 .
drwxr-xr-x 3 root root 4096 Apr 16 18:56 ..
drwxr-xr-x 2 root root 4096 Apr 16 18:56 app
drwxr-xr-x 7 root root 4096 Apr 16 18:56 code
drwxr-xr-x 2 root root 4096 Apr 16 18:56 data
drwxr-xr-x 2 root root 4096 Apr 16 18:56 docs
drwxr-xr-x 8 root root 4096 Apr 16 18:56 .git
-rw-r--r-- 1 root root 1332 Apr 16 18:56 .gitignore
drwxr-xr-x 2 root root 4096 Apr 16 18:56 notebooks
drwxr-xr-x 6 root root 4096 Apr 16 18:56 outpu

In [3]:
# ============================================================
# Cell 3
# ============================================================
import pandas as pd, os

if not os.path.exists("data/train.csv"):
    print("Downloading VLSP 2018 Hotel dataset...")
    !git clone https://github.com/ds4v/absa-vlsp-2018.git /tmp/ds4v --depth=1
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/train.csv data/
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/dev.csv data/
    !cp /tmp/ds4v/datasets/vlsp2018_hotel/test.csv data/
    print("Data downloaded")
else:
    print("Data available")

for split in ["train", "dev", "test"]:
    df = pd.read_csv(f"data/{split}.csv")
    print(f"  {split}: {len(df)} rows × {df.shape[1]} cols")

Data available
  train: 3000 rows × 35 cols
  dev: 2000 rows × 35 cols
  test: 600 rows × 35 cols


In [ ]:
# ============================================================
# Cell 4
# ============================================================
import pandas as pd, os

FORCE_REPROCESS = False

if (not FORCE_REPROCESS) and os.path.exists("data/train_preprocessed.csv"):
    print("Cache available (data/*_preprocessed.csv)")
    s = pd.read_csv("data/train_preprocessed.csv").iloc[0]
    print(f"  Original : {s['Review'][:80]}")
    print(f"  Processed: {str(s.get('processed_review', 'N/A'))[:80]}")
else:
    print("Preprocessing with vncorenlp")
    from step3_preprocessing import preprocess_dataframe, VnCoreNLPSegmenter
    import py_vncorenlp
    vncorenlp_dir = os.path.join(os.getcwd(), 'vncorenlp')
    if not os.path.exists(os.path.join(vncorenlp_dir, 'models', 'wordsegmenter', 'wordsegmenter.rdr')):
        print('Downloading VnCoreNLP models...')
        py_vncorenlp.download_model(save_dir=vncorenlp_dir)
    segmenter = VnCoreNLPSegmenter(vncorenlp_dir=vncorenlp_dir, use_fallback=False)
    for split in ["train", "dev", "test"]:
        df = pd.read_csv(f"data/{split}.csv")
        preprocess_dataframe(df, segmenter=segmenter,
                             cache_path=f"data/{split}_preprocessed.csv")
        print(f"  {split}: {len(df)} rows processed")
    segmenter.close()
    print("Preprocessing completed")

Cache available (data/*_preprocessed.csv)
  Original : Rộng rãi KS mới nhưng rất vắng. Các dịch vụ chất lượng chưa cao và thiếu.
  Processed: Rộng_rãi khách_sạn mới nhưng rất vắng . Các dịch_vụ chất_lượng chưa cao và thiếu


In [5]:
# ============================================================
# Cell 5
# ============================================================
import json, os

enc_cfg = json.load(open('outputs/eda/encoder_config.json'))
print('=== Encoder Config ===')
for k, v in enc_cfg.items():
    print(f'  {k}: {v}')

cw = json.load(open('outputs/eda/class_weights.json'))
print('\n=== Global Class Weights ===')
label_map = {'0': 'absent', '1': 'positive', '2': 'negative', '3': 'neutral'}
for cls, w in cw['global_weights'].items():
    note = '' if float(w) > 10 else ''
    print(f"  {label_map.get(cls,cls):12s}: {float(w):.1f}x{note}")

from utils.constants import TRAIN_CONFIG, ZERO_TRAIN_ASPECTS, PHOBERT_MODEL_NAME

print('\n=== Train Config ===')
for k, v in TRAIN_CONFIG.items():
    print(f'  {k}: {v}')
print(f'  model: {PHOBERT_MODEL_NAME}')

# Verify config values
assert TRAIN_CONFIG['learning_rate'] == 1e-4, f"LR sai: {TRAIN_CONFIG['learning_rate']}"
assert TRAIN_CONFIG['optimizer'] == 'Adam', f"Optimizer sai: {TRAIN_CONFIG['optimizer']}"
assert PHOBERT_MODEL_NAME == 'vinai/phobert-base-v2', f"Model sai: {PHOBERT_MODEL_NAME}"
print(f'\n  ZERO_TRAIN_ASPECTS: {ZERO_TRAIN_ASPECTS}')
print('\nConfig loaded')


=== Encoder Config ===
  recommended_max_seq_len: 256
  p99_word_count: 243
  encoder_option: concat_4_layers
  encoder_hidden_size: 3072
  note: max_seq_len=256

=== Global Class Weights ===
  absent      : 1.0x
  positive    : 8.6x
  negative    : 28.0x
  neutral     : 154.5x

=== Train Config ===
  learning_rate: 0.0001
  warmup_ratio: 0.15
  batch_size: 16
  grad_accumulation_steps: 1
  max_epochs: 20
  early_stop_patience: 7
  dropout: 0.2
  optimizer: Adam
  scheduler: cosine_warmup
  seed: 42
  max_seq_len: 256
  weight_clip: 10.0
  encoder_option: cls_only
  max_grad_norm: 1.0
  model: vinai/phobert-base-v2

  ZERO_TRAIN_ASPECTS: ['ROOM_AMENITIES#PRICES']

Config loaded


In [6]:
# ============================================================
# Cell 6
# ============================================================
import torch
torch.cuda.empty_cache()
print(f"VRAM free before training: {(torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0))/1e9:.1f} GB")

from run_experiment import main

test_metrics = main(encoder_option='concat_4_layers', use_amp=True)

print('\n' + '='*55)
print('MAIN RUN RESULTS')
print(f"  ACD F1:      {test_metrics['macro_acd_f1']:.4f}  (SOTA: 0.8255)")
print(f"  SPC F1:      {test_metrics['macro_spc_f1']:.4f}")
print(f"  Combined F1: {test_metrics['macro_combined_f1']:.4f}  (SOTA: 0.7732)")
print('='*55)


VRAM free before training: 15.6 GB
[Device] GPU: Tesla T4

PHASE PHOBERT — Multi-task ABSA
  encoder:    concat_4_layers
  seq_len:    256
  batch:      16 × 1 = 16 effective
  lr:         0.0001
  weight_clip:10.0
  amp:        ON

[Tokenizer] Loading vinai/phobert-base-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/678 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

...


  Dev — Epoch 13
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    11
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
* FACILITIES#MISCELLANEOUS           0.2000    0.3333    0.1429    0.3333    7
HOTEL#MISCELLANEOUS                  0.2714    0.5000    0.1862    0.1461    145
ROOMS#QUALITY                        0.2857    0.2500    0.3333    0.3333    6
FOOD&DRINKS#PRICES                   0.3019    0.3333    0.2759    0.2593    29
FACILITIES#COMFORT                   0.3061    0.5172    0.2174    0.1905    69
FACILITIES#PRICES                    0.3860    0.5238    0.3056    0.2510    36
FACILITIES#GEN

Train:   0%|          | 0/188 [00:00<?, ?it/s]/kaggle/working/absa-project/code/phobert/train.py:92: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



  Dev — Epoch 14
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    11
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
* FACILITIES#MISCELLANEOUS           0.2222    0.5000    0.1429    0.3333    7
HOTEL#MISCELLANEOUS                  0.2295    0.5526    0.1448    0.1197    145
FOOD&DRINKS#PRICES                   0.3137    0.3636    0.2759    0.2593    29
FACILITIES#COMFORT                   0.3148    0.4359    0.2464    0.2150    69
ROOMS#QUALITY                        0.3333    0.3333    0.3333    0.3333    6
FACILITIES#PRICES                    0.4590    0.5600    0.3889    0.2793    36
ROOM_AMENITIES


  Final — DEV
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    11
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
* FACILITIES#MISCELLANEOUS           0.1250    0.1111    0.1429    0.3333    7
ROOMS#QUALITY                        0.2222    0.1429    0.5000    0.4333    6
FACILITIES#COMFORT                   0.3060    0.2456    0.4058    0.3299    69
HOTEL#MISCELLANEOUS                  0.3308    0.3636    0.3034    0.1952    145
ROOM_AMENITIES#QUALITY               0.3719    0.2681    0.6066    0.4974    122
ROOMS#GENERAL                        0.4115    0.3226    0.5682    0.3202    88
FOOD&DRINKS#PRIC


  Final — TEST
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FACILITIES#MISCELLANEOUS           0.0000    0.0000    0.0000    0.0000    8
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    3
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    4
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    1
HOTEL#MISCELLANEOUS                  0.3276    0.3958    0.2794    0.2013    68
FACILITIES#PRICES                    0.3704    0.3571    0.3846    0.2564    13
FACILITIES#CLEANLINESS               0.3750    0.2727    0.6000    0.3333    5
FACILITIES#GENERAL                   0.4068    0.3158    0.5714    0.2366    21
FACILITIES#COMFORT                   0.4444    0.3478    0.6154    0.3810    26
ROOMS#QUALITY     

In [7]:
# ============================================================
# Cell 7
# ============================================================
# Chứng minh kỹ thuật concat 4 layers có đóng góp thực sự
# (so sánh 3072 dim vs 768 dim trong báo cáo)
# Chạy SAU khi Cell 6 đã hoàn tất

import torch
torch.cuda.empty_cache()

from run_experiment import main

test_metrics_cls = main(encoder_option="cls_only", use_amp=True)

print("\n" + "="*55)
print("ABLATION SUMMARY")
print(f"  concat_4_layers: {test_metrics['macro_combined_f1']:.4f}  ← main architecture")
print(f"  cls_only:        {test_metrics_cls['macro_combined_f1']:.4f}")
gain = test_metrics['macro_combined_f1'] - test_metrics_cls['macro_combined_f1']
print(f"  Gain từ concat:  {gain*100:+.2f}%  {'concat tốt hơn trong lần chạy này' if gain > 0 else 'cls_only tốt hơn trong lần chạy này'}")
print("="*55)

[Device] GPU: Tesla T4

PHASE PHOBERT — Multi-task ABSA
  encoder:    cls_only
  seq_len:    256
  batch:      16 × 1 = 16 effective
  lr:         0.0001
  weight_clip:10.0
  amp:        ON

[Tokenizer] Loading vinai/phobert-base-v2...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[Tokenizer] Loaded 

[Data] Creating DataLoaders...
[DataLoader] train: 3000 samples, 188 batches
[DataLoader] dev: 2000 samples, 125 batches
[DataLoader] test: 600 samples, 38 batches
[Weights] 34 aspects loaded, 77 weight values clipped at 10.0

[Model] Building ABSAPhoBERT (cls_only)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at vinai/phobert-base-v2 and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/kaggle/working/absa-project/code/phobert/train.py:177: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler() if (use_amp and AMP_AVAILABLE and device.type == "cuda") else None



[Model] 135,102,856 trainable parameters
[Scheduler] Total=3760 optimizer steps, Warmup=564
[LR] Adam lr=1.00e-04, scheduler=cosine_warmup
[AMP] Mixed precision: ON
[Config] encoder=cls_only, seq_len=256, batch=16×1=16 (effective)

────────────────────────────────────────────────────────────
Epoch 1/20


...


  Dev — Epoch 18
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    11
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
ROOMS#QUALITY                        0.2105    0.1538    0.3333    0.3333    6
* FACILITIES#MISCELLANEOUS           0.2222    0.5000    0.1429    0.3333    7
HOTEL#MISCELLANEOUS                  0.2781    0.6190    0.1793    0.1527    145
FACILITIES#COMFORT                   0.3486    0.4750    0.2754    0.2891    69
ROOM_AMENITIES#QUALITY               0.4650    0.3802    0.5984    0.4854    122
FACILITIES#PRICES                    0.4762    0.5556    0.4167    0.2862    36
FACILITIES#GE

Train:   0%|          | 0/188 [00:00<?, ?it/s]/kaggle/working/absa-project/code/phobert/train.py:92: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():



  Dev — Epoch 19
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    11
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
ROOMS#QUALITY                        0.2105    0.1538    0.3333    0.3333    6
* FACILITIES#MISCELLANEOUS           0.2222    0.5000    0.1429    0.3333    7
HOTEL#MISCELLANEOUS                  0.2781    0.6190    0.1793    0.1527    145
FACILITIES#COMFORT                   0.3636    0.4878    0.2899    0.3003    69
ROOM_AMENITIES#QUALITY               0.4762    0.3886    0.6148    0.4951    122
FACILITIES#PRICES                    0.4839    0.5769    0.4167    0.2862    36
FACILITIES#GE


  Final — DEV
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    11
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    6
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    7
ROOMS#QUALITY                        0.1176    0.0714    0.3333    0.3333    6
* FACILITIES#MISCELLANEOUS           0.2222    0.5000    0.1429    0.3333    7
HOTEL#MISCELLANEOUS                  0.2842    0.6000    0.1862    0.1570    145
FACILITIES#COMFORT                   0.3548    0.4000    0.3188    0.3085    69
FACILITIES#GENERAL                   0.4430    0.3793    0.5323    0.2418    62
FOOD&DRINKS#PRICES                   0.4590    0.4375    0.4828    0.3810    29
ROOM_AMENITIES#QU


  Final — TEST
Aspect                               ACD-F1    ACD-P     ACD-R     SPC-F1    Support
-----------------------------------  --------  --------  --------  --------  ---------
* FACILITIES#MISCELLANEOUS           0.0000    0.0000    0.0000    0.0000    8
* FOOD&DRINKS#MISCELLANEOUS          0.0000    0.0000    0.0000    0.0000    3
* ROOMS#MISCELLANEOUS                0.0000    0.0000    0.0000    0.0000    4
* ROOM_AMENITIES#MISCELLANEOUS       0.0000    0.0000    0.0000    0.0000    3
* ROOM_AMENITIES#PRICES              0.0000    0.0000    0.0000    0.0000    1
HOTEL#MISCELLANEOUS                  0.2619    0.6875    0.1618    0.1481    68
FACILITIES#CLEANLINESS               0.4000    0.4000    0.4000    0.2667    5
FACILITIES#GENERAL                   0.4483    0.3514    0.6190    0.5699    21
ROOM_AMENITIES#QUALITY               0.4625    0.3663    0.6271    0.4902    59
FACILITIES#PRICES                    0.4667    0.4118    0.5385    0.4048    13
FACILITIES#COMFORT

In [8]:
# ============================================================
# Cell 8
# ============================================================
import json
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

history = json.load(open("outputs/results/training_history.json"))
best_ep = history["best_epoch"]
epochs  = range(1, len(history["train_loss"]) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("PhoBERT concat_4_layers — Learning Curve (ABSA VLSP 2018)", fontsize=13)

# --- Loss ---
ax1.plot(epochs, history["train_loss"], "o-", c="crimson",   lw=2, label="Train Loss")
ax1.plot(epochs, history["dev_loss"],   "o-", c="steelblue", lw=2, label="Dev Loss")
ax1.axvline(best_ep, c="green", ls="--", alpha=0.7, label=f"Best (epoch {best_ep})")
ax1.set(title="Loss", xlabel="Epoch", ylabel="Cross-Entropy Loss")
ax1.legend(); ax1.grid(alpha=0.3)

# --- F1 ---
ax2.plot(epochs, history["dev_acd_f1"],      "s-", c="darkorange", lw=2, label="Dev ACD F1")
ax2.plot(epochs, history["dev_spc_f1"],      "^-", c="purple",     lw=2, label="Dev SPC F1")
ax2.plot(epochs, history["dev_combined_f1"], "o-", c="green",      lw=2.5, label="Dev Combined F1")
ax2.axvline(best_ep, c="green", ls="--", alpha=0.7, label=f"Best (epoch {best_ep})")
ax2.axhline(0.7732,  c="red",   ls=":",  alpha=0.5, label="SOTA Combined 0.7732")
ax2.set(title="F1 Score", xlabel="Epoch", ylabel="Macro F1")
ax2.legend(); ax2.grid(alpha=0.3)

plt.tight_layout()
plt.savefig("outputs/eda/learning_curve.png", dpi=150, bbox_inches="tight")
plt.show()

# --- Phân tích  ---
print(f"\nPhân tích learning curve ():")
print(f"  Best epoch:           {best_ep} / {len(history['train_loss'])}")
print(f"  Best Combined F1:     {history['best_combined_f1']:.4f}")

if len(history["train_loss"]) > best_ep:
    tloss_at   = history["train_loss"][best_ep - 1]
    tloss_after = history["train_loss"][best_ep]
    dloss_at   = history["dev_loss"][best_ep - 1]
    dloss_after = history["dev_loss"][best_ep]
    print(f"  Train loss epoch {best_ep}: {tloss_at:.4f} → epoch {best_ep+1}: {tloss_after:.4f} (tiếp tục giảm)")
    print(f"  Dev loss epoch {best_ep}:   {dloss_at:.4f} → epoch {best_ep+1}: {dloss_after:.4f} (tăng = overfit)")
    print(f"  → Early stopping đúng: dev F1 không cải thiện {history['config']['early_stop_patience']} epoch liên tiếp")

print(f"\n  Saved: outputs/eda/learning_curve.png")


Phân tích learning curve ():
  Best epoch:           7 / 14
  Best Combined F1:     0.4987
  Train loss epoch 7: 0.0801 → epoch 8: 0.0589 (tiếp tục giảm)
  Dev loss epoch 7:   0.1297 → epoch 8: 0.1326 (tăng = overfit)
  → Early stopping đúng: dev F1 không cải thiện 7 epoch liên tiếp

  Saved: outputs/eda/learning_curve.png


In [ ]:
# ============================================================
# Cell 9
# ============================================================
import os, json, shutil

# In summary report
report = "outputs/results/phobert_summary.md"
if os.path.exists(report):
    print(open(report, encoding="utf-8").read())
else:
    print("Chưa có report")

# Liệt kê tất cả files kết quả
print("\n=== Kết quả đã tạo ===")
result_files = []
for root, dirs, files in os.walk("outputs"):
    for f in files:
        if not f.endswith(".DS_Store"):
            path = os.path.join(root, f)
            size = os.path.getsize(path)
            result_files.append(path)
            print(f"  {path} ({size/1024:.1f} KB)")

# Tạo zip để download
print("\nTạo zip...")
shutil.make_archive("/kaggle/working/phobert_results", "zip", "outputs")
print("Zip: /kaggle/working/phobert_results.zip")
print("   → Kaggle: Output panel → Download")

# Tổng hợp kết quả chính
print("\n=== Tổng hợp kết quả ===")
if os.path.exists("outputs/results/phobert_test_metrics.json"):
    m = json.load(open("outputs/results/phobert_test_metrics.json"))
    print(f"PHOBERT_ACD_F1      = {m['macro_acd_f1']:.4f}")
    print(f"PHOBERT_SPC_F1      = {m['macro_spc_f1']:.4f}")
    print(f"PHOBERT_COMBINED_F1 = {m['macro_combined_f1']:.4f}")
if os.path.exists("outputs/results_cls_only/phobert_test_metrics.json"):
    m2 = json.load(open("outputs/results_cls_only/phobert_test_metrics.json"))
    print(f"PHOBERT_ABLATION_COMBINED_F1 = {m2['macro_combined_f1']:.4f}")

# Kết quả PhoBERT Multi-task

## Config thực tế
| Tham số | Giá trị |
|---------|---------|
| Encoder | concat_4_layers |
| MAX_SEQ_LEN | 256 |
| Batch size | 16 × 1 = 16 (effective) |
| Learning rate | 0.0001 |
| Warmup | 15% steps |
| Weight clip | 10.0 |
| Best epoch | 7 / 14 |

## Kết quả

| Split | ACD F1 | SPC F1 | Combined F1 |
|-------|--------|--------|-------------|
| Dev   | 0.5318 | 0.4652 | 0.4985 |
| **Test**  | **0.5419** | **0.4384** | **0.4902** |
| SOTA (Huynh 2022) | 0.8255 | — | 0.7732 |

## Phân tích Gap so với SOTA

- **ACD F1 gap:** 0.2836 (28.4%)
- **Combined F1 gap:** 0.2830 (28.3%)

### Nguyên nhân gap (phân tích):
1. **underthesea vs VnCoreNLP:** Dùng underthesea làm fallback → ~1-2% F1 loss
2. **ROOM_AMENITIES#PRICES:** 0 training samples → ACD F1 = 0 cho aspect này
3. **Neutral cực hiếm (weight=154→clip=10):** SPC F1 cho neutral thấp
4. **Dataset nhỏ (3000 train):** khó học tốt các aspect hiếm
5. **Single model:** báo cáo chính giữ bản single để dễ giải thí